# Brand clustering

Category-level clusters in the brand embedding space.


## Setup


In [ ]:
from pathlib import Path
import os
import re
import warnings
from collections import Counter
from itertools import combinations

CWD = Path.cwd().resolve()
ROOT = CWD
for candidate in [CWD, *CWD.parents]:
    if (candidate / "final_dataset").exists():
        ROOT = candidate
        break


os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".matplotlib_cache"))
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from adjustText import adjust_text
import seaborn as sns
from IPython.display import display
from PIL import Image
from tqdm.auto import tqdm

import sklearn
from sklearn.cluster import AgglomerativeClustering, DBSCAN, HDBSCAN, KMeans
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.metrics.pairwise import cosine_distances, cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder, normalize

try:
    import umap
except Exception:
    umap = None


warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
RNG_SEED = 42
print("Root:", ROOT)
print("UMAP:", getattr(umap, "__version__", "not installed"))
print("scikit-learn:", sklearn.__version__)


repro_path = Path("helpers/reproducibility_helpers.py")
if not repro_path.exists():
    repro_path = Path("../helpers/reproducibility_helpers.py")
if not repro_path.exists():
    repro_path = Path("../../helpers/reproducibility_helpers.py")
if not repro_path.exists():
    repro_path = Path("step_4_clustering/../helpers/reproducibility_helpers.py")
exec(compile(repro_path.read_text(encoding="utf-8"), str(repro_path), "exec"), globals())


## Data

Embeddings aligned with the brand rows.


In [ ]:
EMBEDDING_DIR = ROOT / "step_2_text_embeddings"
if not (EMBEDDING_DIR / "brand_embeddings.npz").exists():
    EMBEDDING_DIR = ROOT / "text_embeddings"
EMBEDDINGS_PATH = EMBEDDING_DIR / "brand_embeddings.npz"
FINAL_DATASET_DIR = ROOT / "final_dataset"

CSV_FILES = {
    "clothes": FINAL_DATASET_DIR / "brands_clothes.csv",
    "shoes": FINAL_DATASET_DIR / "brands_shoes.csv",
    "bags": FINAL_DATASET_DIR / "brands_bags.csv",
    "jewellery": FINAL_DATASET_DIR / "brands_jewellery.csv",
}
VIBE_COLS = ["aesthetic_keywords", "silhouettes", "materials", "palette"]

archive = np.load(EMBEDDINGS_PATH, allow_pickle=True)
row_embeddings = normalize(archive["embeddings"].astype("float32"), norm="l2")
model_name = str(archive["model_name"][0]) if "model_name" in archive.files else "unknown"

frames = []
for category, path in CSV_FILES.items():
    df = pd.read_csv(path)
    df["category"] = category
    frames.append(df)

source_rows = pd.concat(frames, ignore_index=True)
before_drop = len(source_rows)
source_rows = source_rows.dropna(subset=VIBE_COLS, how="all").reset_index(drop=True)
source_rows["row_id"] = np.arange(len(source_rows))

assert len(source_rows) == len(row_embeddings), (
    f"final_dataset rows after vibe-field filtering ({len(source_rows):,}) do not match "
    f"embedding rows ({len(row_embeddings):,}). Embedding files are out of sync with final_dataset."
)

print(f"Loaded {before_drop:,} final_dataset category rows; {len(source_rows):,} after dropping all-empty vibe rows")
print(f"Embedding model: {model_name}")
print(f"Embedding matrix: {row_embeddings.shape}")
display(source_rows["category"].value_counts().rename_axis("category").reset_index(name="rows"))


## Helpers

Clustering helpers and the small hand-labelled validation groups.


In [ ]:
from pathlib import Path

helper_path = Path("helpers/clustering_helpers.py")
if not helper_path.exists():
    helper_path = Path("step_4_clustering/helpers/clustering_helpers.py")
exec(compile(helper_path.read_text(encoding="utf-8"), str(helper_path), "exec"), globals())


## Dataset and validation tables

Counts and validation tables recomputed from the saved dataset and embeddings.


In [ ]:
import runpy

counts_table = dataset_category_counts(ROOT)
validation_family_table = validation_families_table(ROOT)
embedding_validation_table = compute_embedding_validation_summary(ROOT)

save_table(counts_table, "dataset_category_counts", ROOT)
save_table(validation_family_table, "manual_validation_families", ROOT)
save_table(embedding_validation_table, "embedding_validation", ROOT)
embedding_validation_table.to_csv(thesis_figures_dir(ROOT) / "embedding_validation_summary.csv", index=False)
runpy.run_path(str(ROOT / "step_4_clustering" / "helpers" / "figure_exports" / "embedding_validation_figure.py"), run_name="__main__")

print("Dataset counts")
display(counts_table)
print("Manual validation families")
display(validation_family_table)
print("Embedding validation summary")
display(embedding_validation_table)


## Clothes


In [ ]:
clothes_results = run_category_analysis("clothes")


## Shoes


In [ ]:
# Shoes also reports a diagnostic-only sweep beyond k=22 to check whether the selected value is a boundary result.
# Larger k values are not eligible for selection without a minimum-size or stability constraint.
shoes_results = run_category_analysis("shoes")


## Bags


In [ ]:
bags_results = run_category_analysis("bags")


## Jewellery


In [ ]:
jewellery_results = run_category_analysis("jewellery")


## Figure files

Cluster figures and summary tables from the four category results.


In [ ]:
import runpy

cluster_figure_files = runpy.run_path(str(ROOT / "step_4_clustering" / "helpers" / "figure_exports" / "clustering_figures.py"), run_name="__main__")

cluster_selection = cluster_selection_table(ROOT)
cluster_algorithm_comparison = cluster_algorithm_table(ROOT)
save_table(cluster_selection, "cluster_selection_scores", ROOT)
save_table(cluster_algorithm_comparison, "cluster_algorithm_comparison", ROOT)

display(cluster_selection)
display(cluster_algorithm_comparison)
